In [ ]:
!pip install torch torchvision torchaudio
!pip install torch-geometric


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns
import math

from torch_geometric.data import Data
from torch_geometric.nn import GATConv
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

sns.set(style="whitegrid")
pd.set_option("display.max_columns", None)

In [ ]:
df_raw = pd.read_csv(
    "/content/drive/MyDrive/Super_Very_final_anime_dataset.csv",
    engine="python",
    quotechar='"',
    escapechar='\\',
    on_bad_lines="skip"
)
print("Initial Shape:", df_raw.shape)
df_raw.head()


In [ ]:
# Remove missing values
df_raw.dropna(subset=["score", "genre", "user_id", "anime_id", "my_score","Aired"], inplace=True)

# Remove the 'genre_list' column if it exists and contains unhashable types
# This makes the cell robust to re-runs where 'genre_list' might have been created already.
if 'genre_list' in df_raw.columns:
    df_raw.drop(columns=['genre_list'], inplace=True)

# Remove duplicates
df_raw.drop_duplicates(inplace=True)

# Convert genre string → list
df_raw["genre_list"] = df_raw["genre"].apply(lambda x: x.split(",") if isinstance(x, str) else [])

print("After Cleaning:", df_raw.shape)


In [ ]:
from sklearn.preprocessing import MultiLabelBinarizer

#EDA — EXPLORATORY
# Score Distribution
plt.figure(figsize=(8,4))
sns.histplot(df_raw["score"], bins=30, kde=True, color="orchid")
plt.title("Distribution of Anime Scores")
plt.xlabel("Score")
plt.show()

# Popularity distribution
plt.figure(figsize=(8,4))
sns.histplot(df_raw["popularity"], bins=30, kde=True, color="teal")
plt.title("Anime Popularity Distribution")
plt.xlabel("Popularity Rank")
plt.show()

# Top 20 Genres
mlb = MultiLabelBinarizer()
genre_encoded = mlb.fit_transform(df_raw["genre_list"])
genre_counts = pd.DataFrame(genre_encoded, columns=mlb.classes_).sum().sort_values(ascending=False)

plt.figure(figsize=(10,6))
sns.barplot(x=genre_counts.values[:20], y=genre_counts.index[:20], palette="viridis")
plt.title("Top 20 Anime Genres")
plt.xlabel("Count")
plt.ylabel("Genre")
plt.show()

In [ ]:
meta_cols = ["anime_id", "title", "genre", "score", "scored_by", "rank", "popularity","Aired"]
# Add 'year' to meta_cols if it exists in your dataset:
if "year" in df_raw.columns:
    meta_cols.append("Aired")

anime_meta_df = (
    df_raw[meta_cols]
    .drop_duplicates(subset="anime_id")
    .reset_index(drop=True)
)
print("Anime metadata shape:", anime_meta_df.shape)
anime_meta_df.head()

In [ ]:
def get_popular_by_genre(genre_filter, year_filter=None, top_k=5):
    """
    Returns top-k popular anime filtered by genre (and optionally year).

    Parameters:
        genre_filter  : str  — e.g. "Action" or "Romance"
        year_filter   : int  — e.g. 2020 (optional, only if 'year' column exists)
        top_k         : int  — number of recommendations to return
    """
    result = anime_meta_df.copy()

    # Filter by genre (case-insensitive partial match)
    result = result[result["genre"].str.contains(genre_filter, case=False, na=False)]

    # Filter by year if provided and column exists
    if year_filter is not None and "AIRED" in result.columns:
      result = result[result["AIRED"].str.contains(str(year_filter), na=False)]

    if result.empty:
        print(f"No anime found for genre='{genre_filter}'" +
              (f", year={year_filter}" if year_filter else ""))
        return pd.DataFrame()

    # Score = weighted combination of average score + popularity (scored_by)
    # Normalize both to 0-1 range, then combine
    result = result.copy()
    result["norm_score"]  = (result["score"] - result["score"].min()) / \
                             (result["score"].max() - result["score"].min() + 1e-9)
    result["norm_pop"]    = (result["scored_by"] - result["scored_by"].min()) / \
                             (result["scored_by"].max() - result["scored_by"].min() + 1e-9)
    result["hybrid_pop_score"] = 0.6 * result["norm_score"] + 0.4 * result["norm_pop"]

    top_anime = result.sort_values("hybrid_pop_score", ascending=False).head(top_k)

    display_cols = ["title", "genre", "score", "scored_by", "popularity", "hybrid_pop_score"]
    if "year" in top_anime.columns:
        display_cols.insert(2, "year")

    return top_anime[display_cols].reset_index(drop=True)

def recommend_popularity_user(user_id, top_k=10):

    # User history
    user_history = train_df[
        train_df["user_id"] == user_id
    ]

    if user_history.empty:
        return pd.DataFrame()

    # Get watched anime metadata
    watched = anime_meta_df[
        anime_meta_df["anime_id"].isin(
            user_history["anime_id"]
        )
    ]

    if watched.empty:
        return pd.DataFrame()

    # -----------------------------
    # MOST COMMON GENRE
    # -----------------------------
    all_genres = []

    for genres in watched["genre"].dropna():

        split_genres = [
            g.strip()
            for g in str(genres).split(",")
        ]

        all_genres.extend(split_genres)

    if not all_genres:
        return pd.DataFrame()

    favorite_genre = pd.Series(
        all_genres
    ).value_counts().idxmax()

    # -----------------------------
    # PREFERRED YEAR
    # -----------------------------
    watched["year"] = (
        watched["Aired"]
        .astype(str)
        .str.extract(r"(\d{4})")
    )

    watched = watched.dropna(subset=["year"])

    if watched.empty:
        year_filter = None

    else:
        year_filter = int(
            watched["year"]
            .astype(int)
            .median()
        )

    # -----------------------------
    # USE YOUR ORIGINAL FUNCTION
    # -----------------------------
    rec_df = get_popular_by_genre(
        genre_filter=favorite_genre,
        year_filter=year_filter,
        top_k=top_k * 3
    )

    if rec_df.empty:
        return pd.DataFrame()

    # Remove already watched anime
    seen = set(user_history["anime_id"])

    rec_df = rec_df[
        ~rec_df["title"].isin(
            watched["title"]
        )
    ]

    # Merge anime_id back
    rec_df = rec_df.merge(
        anime_meta_df[
            ["anime_id", "title"]
        ],
        on="title",
        how="left"
    )

    return rec_df.head(top_k)[
        [
            "anime_id",
            "title",
            "genre",
            "score",
            "popularity"
        ]
    ]

In [ ]:
print("\n=== Top 5 Popular Action Anime from 2020 ===")
print(get_popular_by_genre("Action", year_filter=2020, top_k=5))

In [ ]:
df = df_raw[["user_id", "anime_id", "my_score", "genre"]].copy()

# Drop rows where my_score is 0 (means user watched but didn't rate)
# You can remove this line if you want to keep them
df = df[df["my_score"] > 0].copy()

print("Interaction rows after removing unrated:", df.shape)

In [ ]:
user_encoder = LabelEncoder()
anime_encoder = LabelEncoder()

df["user_id_enc"]  = user_encoder.fit_transform(df["user_id"])
df["anime_id_enc"] = anime_encoder.fit_transform(df["anime_id"])

num_users = df["user_id_enc"].nunique()
num_anime = df["anime_id_enc"].nunique()

# Shift anime indices so they don't overlap with user indices in the graph
df["anime_id_enc"] = df["anime_id_enc"] + num_users
num_nodes = num_users + num_anime

print(f"Users: {num_users:,} | Anime: {num_anime:,} | Total graph nodes: {num_nodes:,}")


In [ ]:
def train_test_split_userwise(df, test_size=0.2, min_interactions=5):
    train_list, test_list = [], []
    for user in df["user_id_enc"].unique():
        user_data = df[df["user_id_enc"] == user]
        if len(user_data) < min_interactions:
            train_list.append(user_data)  # put cold-start users fully in train
            continue
        train, test = train_test_split(user_data, test_size=test_size, random_state=42)
        train_list.append(train)
        test_list.append(test)
    return pd.concat(train_list).reset_index(drop=True), \
           pd.concat(test_list).reset_index(drop=True)

train_df, test_df = train_test_split_userwise(df)
print(f"Train: {len(train_df):,} | Test: {len(test_df):,}")
print("Building edge_index from train data...")

In [ ]:
import gc

train_users  = torch.tensor(train_df["user_id_enc"].values, dtype=torch.long)
train_animes = torch.tensor(train_df["anime_id_enc"].values, dtype=torch.long)
edge_index = torch.stack([
    torch.cat([train_users, train_animes]),
    torch.cat([train_animes, train_users])
], dim=0)

print(f"Edge_index shape: {edge_index.shape}")  # [2, 2 * num_train_edges]
del train_users, train_animes
gc.collect()

In [ ]:
class GATRecommender(nn.Module):
    def __init__(self, num_nodes, emb_dim=64, hidden_dim=32, out_dim=64,
                 heads=4, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(num_nodes, emb_dim)
        nn.init.xavier_uniform_(self.embedding.weight)

        self.gat1 = GATConv(emb_dim,          hidden_dim, heads=heads,
                             concat=True,  dropout=dropout)
        self.gat2 = GATConv(hidden_dim * heads, out_dim,  heads=1,
                             concat=False, dropout=dropout)
        self.dropout = dropout

    def forward(self, edge_index):
        x = self.embedding.weight                   # [num_nodes, emb_dim]
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.gat1(x, edge_index)
        x = F.elu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.gat2(x, edge_index)
        return x                                    # [num_nodes, out_dim]

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model     = GATRecommender(num_nodes).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.005, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=30, gamma=0.5)

edge_index_gpu = edge_index.to(device)
print("Model parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))

In [ ]:
print("Building user-positive lookup...")
train_user_pos = (
    train_df.groupby("user_id_enc")["anime_id_enc"]
            .apply(set)
            .to_dict()
)
all_anime_enc = np.array(df["anime_id_enc"].unique())

In [ ]:
def bpr_loss(user_emb, pos_emb, neg_emb):
    pos_score = (user_emb * pos_emb).sum(dim=1)
    neg_score = (user_emb * neg_emb).sum(dim=1)
    return -F.logsigmoid(pos_score - neg_score).mean()


def sample_bpr_batch(train_df, batch_size=4096):
    """
    Sample (user, pos_anime, neg_anime) triples.
    neg_anime is drawn uniformly from anime the user has NOT interacted with.
    """
    idx   = np.random.randint(0, len(train_df), batch_size)
    batch = train_df.iloc[idx]

    users     = batch["user_id_enc"].values
    pos_items = batch["anime_id_enc"].values

    # Vectorised negative sampling
    neg_items = np.empty(batch_size, dtype=np.int64)
    for i, u in enumerate(users):
        pos_set = train_user_pos.get(u, set())
        # keep sampling until we get an unseen anime
        neg = np.random.choice(all_anime_enc)
        while neg in pos_set:
            neg = np.random.choice(all_anime_enc)
        neg_items[i] = neg

    users_t     = torch.tensor(users,     dtype=torch.long, device=device)
    pos_items_t = torch.tensor(pos_items, dtype=torch.long, device=device)
    neg_items_t = torch.tensor(neg_items, dtype=torch.long, device=device)

    return users_t, pos_items_t, neg_items_t

In [ ]:
EPOCHS           = 100
BATCH_SIZE       = 4096     # BPR batch size
STEPS_PER_EPOCH  = 50       # how many BPR batches per epoch

print("\nStarting training...")
loss_history = []

for epoch in range(1, EPOCHS + 1):
    model.train()
    optimizer.zero_grad()  # Zero gradients once per epoch

    # One forward pass through the full graph per epoch
    z = model(edge_index_gpu)   # [num_nodes, out_dim]

    total_batch_loss = 0.0 # Initialize accumulator for losses

    for _ in range(STEPS_PER_EPOCH):
        users, pos_items, neg_items = sample_bpr_batch(train_df, BATCH_SIZE)

        loss = bpr_loss(z[users], z[pos_items], z[neg_items])
        total_batch_loss += loss # Accumulate loss

    # After all batches, perform a single backward pass on the aggregated loss
    total_batch_loss.backward()   # No need for retain_graph=True anymore
    optimizer.step()  # Update weights once per epoch

    scheduler.step()
    avg_loss = total_batch_loss.item() / STEPS_PER_EPOCH # Calculate average loss
    loss_history.append(avg_loss)

    if epoch % 10 == 0:
        print(f"Epoch {epoch:3d}/{EPOCHS}  |  BPR Loss: {avg_loss:.4f}  |  LR: {scheduler.get_last_lr()[0]:.5f}")

# Plot training loss
plt.figure(figsize=(8, 4))
plt.plot(loss_history)
plt.xlabel("Epoch")
plt.ylabel("BPR Loss")
plt.title("GAT Training Loss")
plt.tight_layout()
plt.show()

In [ ]:
def recommend_gat(user_id, top_k=5, exclude_seen=True):
    """
    Recommend top-k unseen anime for a given original user_id.

    Parameters:
        user_id      : original user_id value (before encoding)
        top_k        : number of recommendations
        exclude_seen : if True, exclude anime the user has already watched
    """
    model.eval()
    with torch.no_grad():
        z = model(edge_index_gpu).cpu()   # move to CPU for inference

        user_idx = int(user_encoder.transform([user_id])[0])
        user_emb = z[user_idx]            # [out_dim]

        # Anime embeddings: indices num_users .. num_users+num_anime-1
        anime_start = num_users
        anime_end   = num_users + num_anime
        anime_emb   = z[anime_start:anime_end]         # [num_anime, out_dim]

        scores = torch.matmul(anime_emb, user_emb)     # [num_anime]

        # Exclude already-seen anime
        if exclude_seen:
            seen_shifted = set(train_df[train_df["user_id"] == user_id]["anime_id_enc"].values)
            for enc in seen_shifted:
                local_idx = enc - num_users
                if 0 <= local_idx < num_anime:
                    scores[local_idx] = float("-inf")

        top_local = torch.topk(scores, top_k).indices.numpy()
        rec_anime_ids = anime_encoder.inverse_transform(top_local)

    result = anime_meta_df[anime_meta_df["anime_id"].isin(rec_anime_ids)][
        ["anime_id","title", "genre", "score", "popularity"]
    ].reset_index(drop=True)
    return result

In [ ]:
sample_user = df["user_id"].iloc[0]
print(f"\n=== GAT Recommendations for user: {sample_user} ===")
print(recommend_gat(sample_user, top_k=5))

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def build_content_profile(anime_meta_df):
    """
    Creates a 'content_profile' column by concatenating
    genre, type, and source into a single string per anime.
    """
    df = anime_meta_df.copy()

    # Fill missing values
    df["genre"]  = df["genre"].fillna("")
    df["type"]   = df["type"].fillna("") if "type" in df.columns else ""
    df["source"] = df["source"].fillna("") if "source" in df.columns else ""

    # Clean genre: replace commas/hyphens with spaces for TF-IDF tokenisation
    df["genre_clean"] = df["genre"].str.replace(",", " ").str.replace("-", " ").str.strip()

    # Build profile string
    df["content_profile"] = (
        df["genre_clean"] + " " +
        df.get("type",   pd.Series([""] * len(df))).fillna("") + " " +
        df.get("source", pd.Series([""] * len(df))).fillna("")
    ).str.strip()

    return df

anime_content_df = build_content_profile(anime_meta_df)
print("Sample content profiles:")
print(anime_content_df[["title", "content_profile"]].head(5))


In [ ]:
print("\nFitting TF-IDF vectorizer...")

tfidf = TfidfVectorizer(
    ngram_range=(1, 2),    # unigrams + bigrams e.g. "Super Power", "Slice Life"
    min_df=2,              # ignore terms appearing in fewer than 2 anime
    max_features=500       # cap vocabulary size for memory efficiency
)

tfidf_matrix = tfidf.fit_transform(anime_content_df["content_profile"])
print(f"TF-IDF matrix shape: {tfidf_matrix.shape}")

In [ ]:
anime_id_to_idx = {
    aid: idx for idx, aid in enumerate(anime_content_df["anime_id"].values)
}
idx_to_anime_id = {v: k for k, v in anime_id_to_idx.items()}

# Calculate cosine similarity matrix for content-based recommendations
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)
print("Cosine similarity matrix shape:", cosine_sim.shape)


In [ ]:
def recommend_similar_anime(anime_title, top_k=5):
    """
    Given an anime title, returns top-k most similar anime
    based on genre/type/source using cosine similarity.

    Parameters:
        anime_title : str — exact or partial title (case-insensitive)
        top_k       : int — number of recommendations
    """
    # Find matching row (case-insensitive partial match)
    matches = anime_content_df[
        anime_content_df["title"].str.contains(anime_title, case=False, na=False)
    ]
    if matches.empty:
        print(f"No anime found matching '{anime_title}'")
        return pd.DataFrame()

    # Use the first match
    anime_row   = matches.iloc[0]
    anime_id    = anime_row["anime_id"]
    row_idx     = anime_id_to_idx[anime_id]

    print(f"Finding anime similar to: '{anime_row['title']}'")
    print(f"Genres: {anime_row['genre']}\n")

    # Compute cosine similarity between this anime and all others
    anime_vec   = tfidf_matrix[row_idx]                        # sparse [1, vocab]
    sim_scores  = cosine_similarity(anime_vec, tfidf_matrix).flatten()  # [num_anime]

    # Sort by similarity, exclude the query anime itself
    top_indices = np.argsort(sim_scores)[::-1]
    top_indices = [i for i in top_indices if i != row_idx][:top_k]

    results = anime_content_df.iloc[top_indices][
        ["title", "genre", "score", "popularity"]
    ].copy()
    results["similarity"] = sim_scores[top_indices].round(4)
    return results.reset_index(drop=True)

In [ ]:
def recommend_content_for_user(user_id, top_k=5, liked_threshold=7):
    """
    For a given user, finds anime they liked (score >= liked_threshold),
    builds their taste profile by averaging TF-IDF vectors of liked anime,
    then returns top-k most similar unseen anime.

    Parameters:
        user_id           : original user_id
        top_k             : number of recommendations
        liked_threshold   : minimum score to count an anime as "liked"
    """
    user_ratings = train_df[train_df["user_id"] == user_id]

    if user_ratings.empty:
        print(f"No ratings found for user {user_id}")
        return pd.DataFrame()

    # Filter to liked anime
    liked = user_ratings[user_ratings["my_score"] >= liked_threshold]

    if liked.empty:
        print(f"User {user_id} has no anime rated >= {liked_threshold}. Using all rated.")
        liked = user_ratings

    # Get TF-IDF row indices for liked anime
    liked_ids     = liked["anime_id"].values
    liked_indices = [anime_id_to_idx[aid] for aid in liked_ids if aid in anime_id_to_idx]

    if not liked_indices:
        print("None of the user's liked anime are in the content matrix.")
        return pd.DataFrame()

    # Build user taste vector: weighted average of liked anime TF-IDF vectors
    # Weight by (score - threshold) so higher-rated anime matter more
    weights = []
    for aid in liked_ids:
        if aid in anime_id_to_idx:
            score  = liked[liked["anime_id"] == aid]["my_score"].values[0]
            weight = max(score - liked_threshold + 1, 1)
            weights.append(weight)

    liked_matrix  = tfidf_matrix[liked_indices]          # [n_liked, vocab]
    weights_arr   = np.array(weights, dtype=np.float32).reshape(-1, 1)
    # Weighted sum then normalize
    user_profile  = liked_matrix.multiply(weights_arr).sum(axis=0).A1  # Convert to ndarray

    # Cosine similarity against all anime
    sim_scores    = cosine_similarity(user_profile.reshape(1, -1), tfidf_matrix).flatten()

    # Exclude anime the user has already watched
    watched_ids   = set(user_ratings["anime_id"].values)
    watched_indices = {anime_id_to_idx[aid] for aid in watched_ids if aid in anime_id_to_idx}
    for idx in watched_indices:
        sim_scores[idx] = -1   # exclude

    # Top-k
    top_indices = np.argsort(sim_scores)[::-1][:top_k]
    results = anime_content_df.iloc[top_indices][
        ["anime_id","title", "genre", "score", "popularity"]
    ].copy()
    results["similarity"] = sim_scores[top_indices].round(4)
    return results.reset_index(drop=True)

In [ ]:
# --- By anime title ---
print("=== Anime similar to 'Death Note' ===")
print(recommend_similar_anime("Death Note", top_k=5))

print("\n=== Anime similar to 'Naruto' ===")
print(recommend_similar_anime("Naruto", top_k=5))

# --- By user taste ---
sample_user = df_raw["user_id"].iloc[0]
print(f"\n=== Content-Based Recommendations for user: {sample_user} ===")
print(recommend_content_for_user(sample_user, top_k=5, liked_threshold=7))


In [ ]:
def jaccard_sim(set_a, set_b):
    """Jaccard similarity between two genre sets."""
    if not set_a and not set_b:
        return 1.0
    union = len(set_a | set_b)
    return len(set_a & set_b) / union if union > 0 else 0.0

In [ ]:
valid_anime_ids = set(anime_encoder.classes_)

anime_meta_filtered = anime_meta_df[
    anime_meta_df["anime_id"].isin(valid_anime_ids)
].reset_index(drop=True)

In [ ]:
genre_lookup = {
    row["anime_id"]: frozenset(
        g.strip() for g in str(row["genre"]).split(",") if g.strip()
    )
    for _, row in anime_meta_filtered.iterrows()
}

In [ ]:
item_popularity   = train_df.groupby("anime_id")["user_id"].count().to_dict()
total_train_users = train_df["user_id"].nunique()

In [ ]:
def precision_recall_f1_hit_mrr_ndcg(recommended, relevant, k):
    """
    Parameters
    ----------
    recommended : ranked list of anime IDs (best first), len <= k
    relevant    : list/set of ground-truth IDs (rated >= threshold in test)
    k           : cutoff

    Returns
    -------
    precision, recall, f1, hit, mrr, ndcg
    """
    relevant_set  = set(relevant)
    recommended_k = recommended[:k]
    hits = [1 if item in relevant_set else 0 for item in recommended_k]

    precision = sum(hits) / k if k > 0 else 0.0
    recall    = sum(hits) / len(relevant_set) if relevant_set else 0.0
    f1        = (2 * precision * recall / (precision + recall)
                 if (precision + recall) > 0 else 0.0)
    hit       = 1.0 if sum(hits) > 0 else 0.0

    mrr = 0.0
    for rank, item in enumerate(recommended_k, start=1):
        if item in relevant_set:
            mrr = 1.0 / rank
            break

    dcg   = sum(h / math.log2(i + 2) for i, h in enumerate(hits))
    idcg  = sum(1.0 / math.log2(i + 2) for i in range(min(len(relevant_set), k)))
    ndcg  = dcg / idcg if idcg > 0 else 0.0

    return precision, recall, f1, hit, mrr, ndcg


def diversity_at_k(recommended_ids):
    """
    Average pairwise Jaccard DISTANCE between genre sets.
    FIX: receives original-typed anime IDs (not strings).
    """
    if len(recommended_ids) < 2:
        return 0.0
    gs = [genre_lookup.get(aid, frozenset()) for aid in recommended_ids]
    pairs, total = 0, 0.0
    for i in range(len(gs)):
        for j in range(i + 1, len(gs)):
            total += 1.0 - jaccard_sim(gs[i], gs[j])
            pairs += 1
    return total / pairs if pairs > 0 else 0.0


def novelty_at_k(recommended_ids):
    """Average self-information: -log2(p_item). Higher = more novel."""
    if not recommended_ids:
        return 0.0
    novelties = []
    for aid in recommended_ids:
        pop = item_popularity.get(aid, 0)
        p   = pop / total_train_users if total_train_users > 0 else 1e-10
        novelties.append(-math.log2(p + 1e-10))
    return float(np.mean(novelties))


In [ ]:
def mmr_rerank(candidate_ids, candidate_scores, top_k, lambda_mmr=0.65):
    """
    Re-ranks a candidate list using MMR so the final top-K is
    both relevant AND genre-diverse.

    Parameters
    ----------
    candidate_ids    : array/list of anime IDs (large pool, e.g. top-200)
    candidate_scores : relevance score for each candidate (same order)
    top_k            : final list size
    lambda_mmr       : trade-off weight (higher = more relevance-focused)

    Returns
    -------
    selected_ids, selected_scores  (length = top_k)
    """
    candidate_ids    = list(candidate_ids)
    candidate_scores = np.array(candidate_scores, dtype=np.float64)

    # Normalize scores to [0,1]
    lo, hi = candidate_scores.min(), candidate_scores.max()
    norm_scores = (candidate_scores - lo) / (hi - lo + 1e-8)

    genre_sets = [genre_lookup.get(aid, frozenset()) for aid in candidate_ids]

    selected_idx = []
    remaining    = list(range(len(candidate_ids)))

    while len(selected_idx) < top_k and remaining:
        if not selected_idx:
            # Seed with highest relevance item
            best = max(remaining, key=lambda i: norm_scores[i])
        else:
            best_mmr_score = -np.inf
            best = remaining[0]
            for i in remaining:
                rel = norm_scores[i]
                # Maximum genre similarity to already-selected items
                max_sim = max(
                    jaccard_sim(genre_sets[i], genre_sets[j])
                    for j in selected_idx
                )
                score = lambda_mmr * rel - (1.0 - lambda_mmr) * max_sim
                if score > best_mmr_score:
                    best_mmr_score = score
                    best = i
        selected_idx.append(best)
        remaining.remove(best)

    sel_ids    = [candidate_ids[i]    for i in selected_idx]
    sel_scores = [candidate_scores[i] for i in selected_idx]
    return sel_ids, sel_scores


In [ ]:
import math
#hybrid function
POOL_SIZE  = 200    # candidate pool before MMR re-ranking
LAMBDA_MMR = 0.65   # MMR trade-off (tune between 0.5–0.8)

def hybrid_recommend_final(user_id, top_k=20, exclude_seen=True):
    """
    Improved hybrid recommender with MMR re-ranking.
    Returns a DataFrame with top_k diverse, relevant anime.
    """
    model.eval()
    with torch.no_grad():
        z        = model(edge_index_gpu).cpu()
        user_idx = int(user_encoder.transform([user_id])[0])
        user_emb = z[user_idx]
        anime_emb = z[num_users : num_users + num_anime]

        # ── 1. GAT CF scores (encoder-aligned) ──────────────────────────
        cf_scores = torch.matmul(anime_emb, user_emb).numpy()   # (num_anime,)

        # ── 2. Content scores (encoder-aligned) ─────────────────────────
        user_anime_ids  = df[df["user_id"] == user_id]["anime_id"]
        user_genre_set  = set()
        for aid in user_anime_ids:
            user_genre_set |= genre_lookup.get(aid, frozenset())

        meta_lookup_idx = anime_meta_filtered.set_index("anime_id")
        classes         = anime_encoder.classes_

        content_scores    = np.zeros(num_anime, dtype=np.float32)
        popularity_scores = np.zeros(num_anime, dtype=np.float32)

        for i, aid in enumerate(classes):
            if aid in meta_lookup_idx.index:
                row = meta_lookup_idx.loc[aid]
                content_scores[i]    = len(user_genre_set & genre_lookup.get(aid, frozenset()))
                popularity_scores[i] = 1.0 / (float(row["popularity"]) + 1.0)

        # ── 3. Normalize ─────────────────────────────────────────────────
        def norm(x):
            return (x - x.min()) / (x.max() - x.min() + 1e-8)

        cf_scores         = norm(cf_scores)
        content_scores    = norm(content_scores)
        popularity_scores = norm(popularity_scores)

        # Step: Keep only strong CF candidates
        cf_threshold = np.percentile(cf_scores, 50)  # top 30%

        mask = cf_scores >= cf_threshold

        cf_scores = np.where(mask, cf_scores, 0)
        content_scores = np.where(mask, content_scores, 0)
        popularity_scores = np.where(mask, popularity_scores, 0)

        # ── 4. Hybrid score — reduced popularity weight ──────────────────
        #   Old: 0.5 CF + 0.3 content + 0.2 popularity  → popularity bias
        #   New: 0.6 CF + 0.35 content + 0.05 popularity → less bias
        final_scores = (
            0.60 * cf_scores +
            0.3 * content_scores +
            0.1 * popularity_scores)


        # ── 5. Exclude TRAINING-SEEN anime only (not test!) ─────────────
        if exclude_seen:
            seen_shifted = set(train_df[train_df["user_id"] == user_id]["anime_id_enc"].values)
            for enc in seen_shifted:
                local_idx = int(enc) - num_users
                if 0 <= local_idx < num_anime:
                    final_scores[local_idx] = -np.inf

        # ── Threshold filtering (reduce noise) ─────────────────────
        threshold = 0.1   # start safe, adjust if needed

        mask = final_scores > threshold
        final_scores = np.where(mask, final_scores, -np.inf)

        #valid_scores = final_scores[np.isfinite(final_scores)]
        #print("Min:", valid_scores.min())
        #print("Max:", valid_scores.max())

        # ── 6. Large candidate pool ──────────────────────────────────────
        pool_size    = min(POOL_SIZE, int(np.isfinite(final_scores).sum()))
        pool_indices = np.argsort(final_scores)[-pool_size:][::-1]
        pool_ids     = classes[pool_indices]
        pool_scores  = final_scores[pool_indices]

        # ── 7. MMR re-rank for diversity ─────────────────────────────────
        reranked_ids, reranked_scores = mmr_rerank(
            pool_ids, pool_scores, top_k=top_k, lambda_mmr=LAMBDA_MMR
        )

        result = pd.DataFrame({
            "anime_id":   reranked_ids,
            "score_pred": reranked_scores
        })
        result = result.merge(anime_meta_filtered, on="anime_id", how="left")
        return result[["anime_id", "title", "genre", "score", "popularity"]]

In [ ]:
sample_user = train_df["user_id"].iloc[0]
hybrid_recommend_final(sample_user, top_k=10)

In [ ]:
def recommend_popularity(user_id, top_k=10):
    """
    Recommend globally popular anime.
    Excludes anime already seen by user.
    """

    seen = set(
        train_df[train_df["user_id"] == user_id]["anime_id"]
    )

    pop_df = anime_meta_filtered.copy()

    # popularity rank: lower = better
    pop_df = pop_df.sort_values(
        by=["score", "scored_by"],
        ascending=False
    )

    pop_df = pop_df[
        ~pop_df["anime_id"].isin(seen)
    ]

    return pop_df.head(top_k)[
        ["anime_id", "title", "genre", "score", "popularity"]
    ]

In [ ]:
RELEVANCE_THRESHOLD = 7   # minimum my_score to count as "relevant"

def evaluate_hybrid(k=10, num_users_eval=50, rel_threshold=RELEVANCE_THRESHOLD):
    """
    Evaluate the hybrid recommender on held-out test users.

    Parameters
    ----------
    k              : recommendation list length
    num_users_eval : number of test users to evaluate
    rel_threshold  : minimum score in test set to count as relevant (default 7)
    """
    users = list(set(test_df["user_id"]) & set(train_df["user_id"]))[:num_users_eval]

    results         = []
    all_recommended = []

    for user in users:
        valid_anime  = set(anime_encoder.classes_)
        test_user_df = test_df[test_df["user_id"] == user]

        # Only high-scoring test items are "relevant"
        relevant_ids = [
            aid for aid, sc in zip(
                test_user_df["anime_id"].tolist(),
                test_user_df["my_score"].tolist()
            )
            if aid in valid_anime and sc >= rel_threshold
        ]

        if not relevant_ids:
            continue

        try:
            rec_df = hybrid_recommend_final(user, top_k=k)
        except Exception as e:
            print(f"[WARN] user {user}: {e}")
            continue

        if rec_df.empty:
            continue

        # Keep original types for diversity / novelty
        recommended_orig = rec_df["anime_id"].tolist()
        all_recommended.extend(recommended_orig)

        # Convert to str for safe set intersection
        rec_str = [str(x) for x in recommended_orig]
        rel_str = [str(x) for x in relevant_ids]

        precision, recall, f1, hit, mrr, ndcg = precision_recall_f1_hit_mrr_ndcg(
            rec_str, rel_str, k
        )
        diversity = diversity_at_k(recommended_orig)   # original type ← BUG FIX
        novelty   = novelty_at_k(recommended_orig)

        results.append({
            "precision": precision,
            "recall":    recall,
            "f1":        f1,
            "hit_rate":  hit,
            "mrr":       mrr,
            "ndcg":      ndcg,
            "diversity": diversity,
            "novelty":   novelty,
        })

    if not results:
        print("[ERROR] No valid users evaluated.")
        return pd.DataFrame()

    df_res = pd.DataFrame(results)

    total_items = anime_meta_filtered["anime_id"].nunique()
    coverage    = len(set(str(x) for x in all_recommended)) / total_items

    means = df_res.mean()
    stds  = df_res.std()

    print(f"\n{'='*50}")
    print(f"  Evaluation @K={k}  |  {len(results)} users  |  rel_threshold≥{rel_threshold}")
    print(f"{'='*50}")
    print(f"  {'Metric':<12} {'Mean':>10}  {'±Std':>10}")
    print(f"  {'-'*36}")
    for col in df_res.columns:
        print(f"  {col:<12} {means[col]:>10.4f}  {stds[col]:>10.4f}")
    print(f"  {'coverage':<12} {coverage:>10.4f}")
    print(f"{'='*50}\n")

    summary = means.to_frame(name="mean")
    summary["std"] = stds
    summary.loc["coverage", "mean"] = coverage
    summary.loc["coverage", "std"]  = 0.0
    return summary


In [ ]:
def evaluate_model(
    recommender_function,
    model_name="Model",
    k=10,
    num_users_eval=100,
    rel_threshold=7
):

    users = list(
        set(test_df["user_id"]) &
        set(train_df["user_id"])
    )[:num_users_eval]

    results = []
    all_recommended = []

    for user in users:

        test_user_df = test_df[
            test_df["user_id"] == user
        ]

        relevant_ids = [
            aid for aid, sc in zip(
                test_user_df["anime_id"],
                test_user_df["my_score"]
            )
            if sc >= rel_threshold
        ]

        if not relevant_ids:
            continue

        try:

            rec_df = recommender_function(
                user,
                top_k=k
            )

        except Exception as e:
            print(f"{model_name} error: {e}")
            continue

        if rec_df.empty:
            continue

        recommended_ids = rec_df["anime_id"].tolist()

        all_recommended.extend(recommended_ids)

        precision, recall, f1, hit, mrr, ndcg = \
            precision_recall_f1_hit_mrr_ndcg(
                recommended_ids,
                relevant_ids,
                k
            )

        diversity = diversity_at_k(
            recommended_ids
        )

        novelty = novelty_at_k(
            recommended_ids
        )

        results.append({
            "precision": precision,
            "recall": recall,
            "f1": f1,
            "hit_rate": hit,
            "mrr": mrr,
            "ndcg": ndcg,
            "diversity": diversity,
            "novelty": novelty
        })

    df_res = pd.DataFrame(results)

    means = df_res.mean()

    coverage = len(set(all_recommended)) / \
        anime_meta_filtered["anime_id"].nunique()

    means["coverage"] = coverage

    print(f"\n===== {model_name} =====")
    print(means)

    return means

In [ ]:
pop_results = evaluate_model(
    recommend_popularity_user,
    model_name="Popularity System"
)

tfidf_results = evaluate_model(
    recommend_content_for_user,
    model_name="TF-IDF Content-Based"
)

gat_results = evaluate_model(
    recommend_gat,
    model_name="GAT Collaborative Filtering"
)

hybrid_results = evaluate_model(
    hybrid_recommend_final,
    model_name="Hybrid + MMR"
)

/tmp/ipykernel_6122/708734266.py:85: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  watched["year"] = (
/tmp/ipykernel_6122/708734266.py:85: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  watched["year"] = (
/tmp/ipykernel_6122/708734266.py:85: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-


===== Popularity System =====
precision    0.198000
recall       0.069807
f1           0.086691
hit_rate     0.810000
mrr          0.427190
ndcg         0.219058
diversity    0.690573
novelty      1.489081
coverage     0.006854
dtype: float64

===== TF-IDF Content-Based =====
precision    0.039000
recall       0.011807
f1           0.015008
hit_rate     0.260000
mrr          0.116675
ndcg         0.043948
diversity    0.246006
novelty      5.452826
coverage     0.037638
dtype: float64

===== GAT Collaborative Filtering =====
precision    0.221000
recall       0.082683
f1           0.100399
hit_rate     0.800000
mrr          0.518274
ndcg         0.255089
diversity    0.730649
novelty      1.269304
coverage     0.004209
dtype: float64

===== Hybrid + MMR =====
precision    0.150000
recall       0.061181
f1           0.071510
hit_rate     0.750000
mrr          0.467885
ndcg         0.188818
diversity    0.750017
novelty      1.832612
coverage     0.004930
dtype: float64


In [ ]:
def sanity_check_overlap(num_check=5, k=10, rel_threshold=RELEVANCE_THRESHOLD):
    users = list(set(test_df["user_id"]) & set(train_df["user_id"]))[:num_check]
    print(f"\nSanity check (rel_threshold >= {rel_threshold}):")
    print(f"{'User':>12} | {'relevant':>8} | {'recommended':>11} | {'overlap':>7}")
    print("-" * 48)
    for user in users:
        test_user = test_df[test_df["user_id"] == user]
        relevant  = set(
            str(aid) for aid, sc in zip(
                test_user["anime_id"].tolist(), test_user["my_score"].tolist()
            ) if sc >= rel_threshold
        )
        rec_df      = hybrid_recommend_final(user, top_k=k)
        recommended = set(str(x) for x in rec_df["anime_id"].tolist())
        print(f"{str(user):>12} | {len(relevant):>8} | {len(recommended):>11} | {len(relevant & recommended):>7}")

print("Running sanity check...")
sanity_check_overlap(num_check=5, k=20)

print("\n===== POPULARITY SYSTEM =====")
pop_results = evaluate_model(
    recommend_popularity_user,
    model_name="Popularity System",
    k=10
)

print("\n===== TF-IDF CONTENT SYSTEM =====")
tfidf_results = evaluate_model(
    recommend_content_for_user,
    model_name="TF-IDF Content-Based",
    k=10
)

print("\n===== GAT COLLABORATIVE FILTERING =====")
gat_results = evaluate_model(
    recommend_gat,
    model_name="GAT Collaborative Filtering",
    k=10
)

print("\n===== HYBRID + MMR SYSTEM =====")
hybrid_results = evaluate_model(
    hybrid_recommend_final,
    model_name="Hybrid + MMR",
    k=10
)

In [ ]:
comparison_table = pd.DataFrame({

    "Popularity": pop_results,

    "TF-IDF": tfidf_results,

    "GAT": gat_results,

    "Hybrid+MMR": hybrid_results

}).T

print(comparison_table.round(4))

GRADIO UI SECTION

In [ ]:
!pip install gradio

In [ ]:
import gradio as gr
import pandas as pd

# =========================================================
# 🎌 PREPARE UI DATA
# =========================================================

df_ui = anime_meta_df.copy()

# Extract year
df_ui["year"] = (
    df_ui["Aired"]
    .astype(str)
    .str.extract(r"(\d{4})")
    .fillna(0)
    .astype(int)
)

# Clean genres
df_ui["cleaned_genres"] = df_ui["genre"].apply(
    lambda x: [g.strip() for g in str(x).split(",") if g.strip()]
)

# Merge Image URLs
if "Image URL" in df_raw.columns and "anime_id" in df_raw.columns:

    img_df = df_raw[
        ["anime_id", "Image URL"]
    ].drop_duplicates("anime_id")

    df_ui = pd.merge(
        df_ui,
        img_df,
        on="anime_id",
        how="left"
    )

else:
    df_ui["Image URL"] = ""

# Rating out of 5
df_ui["rating"] = (df_ui["score"] / 2).round(1)

# All genres
all_genres = sorted({
    g for sublist in df_ui["cleaned_genres"]
    for g in sublist if g
})

# =========================================================
# ⚡ FAST LOOKUPS
# =========================================================

image_lookup = (
    df_ui
    .set_index("anime_id")["Image URL"]
    .to_dict()
)

year_lookup = (
    df_ui
    .set_index("anime_id")["year"]
    .to_dict()
)

# =========================================================
# 🎨 CARD CSS
# =========================================================

CARD_CSS = """
<style>

.ag-grid{
    display:flex;
    flex-wrap:wrap;
    gap:18px;
    padding:10px 0;
}

.ag-card{
    width:170px;
    border-radius:16px;
    padding:10px;
    background:linear-gradient(145deg,#1a1a2e,#16213e);
    color:white;
    box-shadow:0 6px 18px rgba(0,0,0,.4);
    transition:0.25s;
}

.ag-card:hover{
    transform:translateY(-6px);
}

.ag-card img{
    width:100%;
    height:220px;
    object-fit:cover;
    border-radius:10px;
}

.ag-title{
    font-size:13px;
    font-weight:700;
    margin-top:8px;
    color:#dfe6ff;
}

.ag-rating{
    color:#ffd700;
    font-size:12px;
    margin-top:5px;
}

.ag-meta{
    color:#9fb0d9;
    font-size:11px;
    margin-top:4px;
}

.ag-genre{
    color:#8db8ff;
    font-size:10px;
    margin-top:5px;
}

.ag-badge{
    display:inline-block;
    background:#7f5af0;
    padding:3px 8px;
    border-radius:20px;
    font-size:10px;
    margin-top:5px;
}

</style>
"""

# =========================================================
# 🎴 REUSABLE CARD BUILDER
# =========================================================

def build_cards(df_result, badge_col=None, badge_label=""):

    if df_result is None or df_result.empty:
        return "<h3 style='color:white'>No anime found 😢</h3>"

    html = CARD_CSS + '<div class="ag-grid">'

    for _, row in df_result.iterrows():

        anime_id = row.get("anime_id", None)

        title = str(row.get("title", "Unknown"))[:40]

        score = float(row.get("score", 0) or 0)

        rating = round(score / 2, 1)

        stars = "★" * int(rating) + "☆" * (5 - int(rating))

        genres = row.get("genre", "")

        if isinstance(genres, list):
            genres = ", ".join(genres[:3])
        else:
            genres = ", ".join(str(genres).split(",")[:3])

        year = year_lookup.get(anime_id, row.get("year", "N/A"))

        image_url = image_lookup.get(anime_id, "")

        badge_html = ""

        if badge_col and badge_col in row:

            badge_html = f"""
            <div class="ag-badge">
                {badge_label}: {round(float(row[badge_col]),3)}
            </div>
            """

        html += f"""

        <div class="ag-card">

            <img src="{image_url}">

            <div class="ag-title">
                {title}
            </div>

            <div class="ag-rating">
                {stars} {rating}/5
            </div>

            <div class="ag-meta">
                📅 {year}
            </div>

            {badge_html}

            <div class="ag-genre">
                🎯 {genres}
            </div>

        </div>

        """

    html += "</div>"

    return html

# =========================================================
# 🔍 SIMILAR ANIME TAB
# =========================================================

def ui_similar(anime_name, top_k):

    try:

        anime_name = anime_name.strip()

        if not anime_name:
            return "⚠️ Please enter anime name.", ""

        result = recommend_similar_anime(
            anime_name,
            top_k=int(top_k)
        )

        if result is None or result.empty:
            return f"😢 No anime found similar to '{anime_name}'", ""

        # Merge anime_id
        result = pd.merge(
            result,
            df_ui[
                ["title", "anime_id"]
            ].drop_duplicates("title"),
            on="title",
            how="left"
        )

        status = (
            f"🔍 Found {len(result)} anime similar to '{anime_name}'"
        )

        return status, build_cards(
            result,
            badge_col="similarity",
            badge_label="Similarity"
        )

    except Exception as e:
        return f"❌ Error: {str(e)}", ""

# =========================================================
# 📊 BROWSE TAB
# =========================================================

def ui_browse(genres, year):

    try:

        if not genres:
            return "⚠️ Please select at least one genre.", ""

        if len(genres) > 4:
            return "⚠️ Max 4 genres allowed.", ""

        year = int(float(year))

        data = df_ui.copy()

        # Year filter
        year_filtered = data[
            data["year"] == year
        ]

        if not year_filtered.empty:
            data = year_filtered

        # Genre filter
        genre_filtered = data[
            data["cleaned_genres"].apply(
                lambda x: any(g in x for g in genres)
            )
        ]

        if not genre_filtered.empty:
            data = genre_filtered

        data = (
            data
            .sort_values(
                by="rating",
                ascending=False
            )
            .head(20)
        )

        status = f"""
        📊 Top {len(data)} Anime

        Genres: {', '.join(genres)}

        Year: {year}
        """

        return status, build_cards(data)

    except Exception as e:
        return f"❌ Error: {str(e)}", ""

# =========================================================
# 🎨 MAIN CSS
# =========================================================

MAIN_CSS = """

.gradio-container{
    background:#0b1020 !important;
    font-family:'Segoe UI',sans-serif;
}

.gr-button-primary{
    background:linear-gradient(
        135deg,
        #5b5ef4,
        #a855f7
    ) !important;

    border:none !important;
    color:white !important;
}

.gr-button-primary:hover{
    opacity:0.9;
}

label,h1,h2,h3,p{
    color:#dfe6ff !important;
}

textarea,input{
    background:#151530 !important;
    color:white !important;
}

"""

# =========================================================
# 🚀 FINAL APP
# =========================================================

with gr.Blocks(
    css=MAIN_CSS,
    title="Anime Recommendation System"
) as demo:

    gr.Markdown("# Anime Recommendation System")

    with gr.Tabs():

        # =====================================================
        # 🔍 SIMILAR ANIME TAB
        # =====================================================

        with gr.Tab("🔍 Similar Anime"):

            with gr.Row():

                anime_input = gr.Textbox(
                    label="Anime Name",
                    placeholder="Naruto, Death Note..."
                )

                topk_sim = gr.Slider(
                    5,
                    50,
                    value=20,
                    step=5,
                    label="Top-K"
                )

            sim_btn = gr.Button(
                "Find Similar Anime",
                variant="primary"
            )

            sim_status = gr.Textbox(
                label="Status",
                interactive=False
            )

            sim_html = gr.HTML()

            sim_btn.click(
                fn=ui_similar,
                inputs=[
                    anime_input,
                    topk_sim
                ],
                outputs=[
                    sim_status,
                    sim_html
                ]
            )

        # =====================================================
        # 📊 BROWSE TAB
        # =====================================================

        with gr.Tab("📊 Browse by Genre"):

            with gr.Row():

                genre_input = gr.Dropdown(
                    choices=all_genres,
                    multiselect=True,
                    label="Genres (Max 4)"
                )

                year_input = gr.Number(
                    label="Year",
                    value=2020
                )

            browse_btn = gr.Button(
                "Browse Anime",
                variant="primary"
            )

            browse_status = gr.Textbox(
                label="Status",
                interactive=False
            )

            browse_html = gr.HTML()

            browse_btn.click(
                fn=ui_browse,
                inputs=[
                    genre_input,
                    year_input
                ],
                outputs=[
                    browse_status,
                    browse_html
                ]
            )

# =========================================================
# 🚀 LAUNCH
# =========================================================

demo.launch(
    share=True,
    debug=True
)

/tmp/ipykernel_6122/699248317.py:357: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://1573d372ba5ea8beb8.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Finding anime similar to: 'One Piece'
Genres: Action, Adventure, Comedy, Super Power, Drama, Fantasy, Shounen

Finding anime similar to: 'Naruto: Shippuuden'
Genres: Action, Adventure, Comedy, Super Power, Martial Arts, Shounen

No anime found matching 'dormaon'
Finding anime similar to: 'Death Note'
Genres: Mystery, Police, Psychological, Supernatural, Thriller, Shounen

